# PharmIQ — System 3: Therapeutic Category Classifier
Rule-seeded label engineering → LightGBM multiclass classifier. AUC 0.999, Accuracy 96%.

In [3]:
import os, sys, warnings, joblib
os.chdir(r'c:\Users\faffo\Project\ML_PharmIQ')
sys.path.insert(0, os.getcwd())
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.ingest import load_raw, clean
from src.features.engineer import extract_dosage_form, extract_salt_count
from src.classifier.label_engine import (
    assign_label, coverage_report, CATEGORY_CODES, CODE_TO_CATEGORY, PRIORITY_ORDER
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

df = clean(load_raw('data/raw/tata_1mg_Medicine_data.csv'))
df['dosage_form'] = df['Quantity'].apply(extract_dosage_form)
df['salt_count'] = df['Salt_Composition'].apply(extract_salt_count)
df['category'] = df['Salt_Composition'].apply(assign_label)
df['label_code'] = df['category'].map(CATEGORY_CODES)

pipeline = joblib.load('models/category_classifier_v1.pkl')
print('Ready.')

Ready.


## 1. Label Coverage

In [4]:
report = coverage_report(df['Salt_Composition'].tolist())
print(f'Coverage: {report["coverage_pct"]}% ({report["labelled"]:,} / {report["total"]:,})')
print()
for cat, count in report['distribution'].items():
    print(f'  {cat:<25}: {count:>7,}')

Coverage: 82.47% (225,694 / 273,655)

  Antibiotic               :  58,384
  Other                    :  47,961
  Analgesic                :  36,898
  Gastrointestinal         :  27,181
  Cardiac                  :  20,971
  Respiratory              :  19,380
  Neurological             :  16,398
  Anti-diabetic            :  16,146
  Dermatology              :  10,285
  Hormonal                 :   9,315
  Vitamin/Supplement       :   6,511
  Anti-parasitic           :   2,266
  Musculoskeletal          :   1,262
  Ophthalmic               :     697


## 2. Evaluate on Test Set

In [5]:
df_lab = df[df['category'] != 'Other'].copy()
X = df_lab[['Salt_Composition','dosage_form','salt_count']]
y = df_lab['label_code']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
print(f'Accuracy : {acc:.4f}')
print(f'AUC OvR  : {auc:.4f}')

present_codes = sorted(y_test.unique())
present_names = [CODE_TO_CATEGORY[c] for c in present_codes]
print(classification_report(y_test, y_pred, labels=present_codes, target_names=present_names))

Accuracy : 0.9601
AUC OvR  : 0.9990
                    precision    recall  f1-score   support

     Anti-diabetic       1.00      0.97      0.98      3229
        Antibiotic       1.00      0.97      0.99     11677
           Cardiac       0.99      0.93      0.96      4194
      Neurological       0.93      0.93      0.93      3280
          Hormonal       0.94      0.92      0.93      1863
    Anti-parasitic       0.66      0.95      0.78       453
        Ophthalmic       0.28      0.92      0.43       139
         Analgesic       1.00      0.97      0.99      7380
       Respiratory       1.00      0.97      0.98      3876
  Gastrointestinal       0.99      0.96      0.98      5436
Vitamin/Supplement       0.98      0.99      0.99      1302
       Dermatology       0.95      0.95      0.95      2057
   Musculoskeletal       0.26      0.93      0.40       253

          accuracy                           0.96     45139
         macro avg       0.84      0.95      0.87     45139
  

## 3. Rule vs Model Agreement

In [6]:
# On 'Other' (unlabelled) medicines — model predicts, rule returns Other
df_other = df[df['category'] == 'Other'].copy()
X_other = df_other[['Salt_Composition','dosage_form','salt_count']].head(5000)
preds_other = pipeline.predict(X_other)
pred_cats = [CODE_TO_CATEGORY.get(int(p),'Other') for p in preds_other]
dist = pd.Series(pred_cats).value_counts()
print('Model predictions for rule-unlabelled medicines:')
print(dist.to_string())

Model predictions for rule-unlabelled medicines:
Anti-parasitic        897
Neurological          765
Ophthalmic            754
Musculoskeletal       611
Hormonal              556
Gastrointestinal      437
Dermatology           307
Cardiac               279
Antibiotic            143
Respiratory           135
Analgesic              54
Vitamin/Supplement     34
Anti-diabetic          28
